In [2]:
import cv2
import numpy as np
import imageio
import time  # <-- 1. Import time
from IPython.display import display, Video
from matplotlib import pyplot as plt
from tqdm import trange
from flygym.compose import ActuatorType
from miniproject.simulation import MiniprojectSimulation
from submission.controller import Controller

sim = MiniprojectSimulation(level=4, seed=67)  ## pb avec seed 10 , 15 et 777 (collines)
controller = Controller(sim)

# 2. Create a UNIQUE filename using a timestamp to bypass browser caching
timestamp = int(time.time())
video_filename = f"fly_vision_contours_{timestamp}.mp4"

# On choisit un framerate de 30 fps
writer = imageio.get_writer(video_filename, fps=30)

for step in trange(50000):
    joint_angles, adhesion, contours, best_contour, dragonfly_contours, state, forces = controller.step(sim)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
    sim.step()
    sim.render_as_needed()
    
    # Capturer la vision
    if step % 20 == 0:
        
        raw_vision = sim.get_raw_vision(sim.fly.name)
        
        left_eye = raw_vision[0].copy()
        right_eye = raw_vision[1].copy()
        
        combined_vision = np.hstack((left_eye, right_eye))
        
        for c in contours:
            try:
                cv2.drawContours(combined_vision, [c[0]], -1, (0, 0, 255), 2)
            except Exception as e:
                pass

        for c in dragonfly_contours:
            try:
                cv2.drawContours(combined_vision, [c], -1, (0, 255, 0), 2)
            except Exception as e:
                pass

        #affiche le state en haut à gauche     
        try :
            cv2.addText(combined_vision, state)
        except Exception as e:
            pass 

        if best_contour is not None:
            try:
                cv2.drawContours(combined_vision, [best_contour[0]], -1, (255, 0, 0), 2)
            except Exception as e:
                pass

        # debuggage vent
        #data = sim.get_antenna_data(sim.fly.name)
        #qpos_l = data['l']['qpos']
        #theta_l = 2*np.acos(qpos_l[0])
        #axis_of_rotation_l = qpos_l[1:]/np.sin(theta_l/2)
        #norm_l = np.linalg.norm(axis_of_rotation_l)
        #print(f"Left rotation_angle", theta_l, "axis:", axis_of_rotation_l)
        #qpos_r = data['r']['qpos']
        #theta_r = 2*np.acos(qpos_r[0])
        #axis_of_rotation_r = qpos_r[1:]/np.sin(theta_r/2)
        #norm_r = np.linalg.norm(axis_of_rotation_r)
        #print(f"Right rotation_angle", theta_r, "axis:", axis_of_rotation_r)

        # Convert from BGR (OpenCV) to RGB (Video)
        
        
        writer.append_data(combined_vision)

# Fermer le fichier vidéo
writer.close()

# Afficher le rendu global de la simulation
sim.renderer.show_in_notebook()

# 3. Afficher la vidéo avec le NOUVEAU nom de fichier unique
display(Video(video_filename, embed=True, width=800))

  0%|          | 0/50000 [00:00<?, ?it/s]

Leg 4 early touchdown! Force: 4.46, Phase: 5.10
Leg 5 early touchdown! Force: 1.58, Phase: 5.73


ValueError: too many values to unpack (expected 7)

In [ ]:
import cv2
import numpy as np
import imageio
import time
from IPython.display import display, Video
from matplotlib import pyplot as plt
from tqdm import trange
from flygym.compose import ActuatorType
from miniproject.simulation import MiniprojectSimulation
from submission.controller import Controller

sim = MiniprojectSimulation(level=4, seed=67)
controller = Controller(sim)

timestamp = int(time.time())
video_filename = f"fly_vision_dashboard_{timestamp}.mp4"

writer = imageio.get_writer(video_filename, fps=30)

# --- Configuration du panneau de debug ---
PANEL_WIDTH = 400
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.45
FONT_COLOR = (255, 255, 255)
LINE_SPACING = 20

for step in trange(50000):
    # ATTENTION: Assure-toi que ton controller.step renvoie bien le 'drive' en 7ème position
    # Si le drive est stocké ailleurs (ex: controller.drive), retire-le de cette ligne
    joint_angles, adhesion, contours, best_contour, dragonfly_contours, state, drive, forces = controller.step(sim)
    
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
    sim.step()
    sim.render_as_needed()
    
    if step % 20 == 0:
        raw_vision = sim.get_raw_vision(sim.fly.name)
        left_eye = raw_vision[0].copy()
        right_eye = raw_vision[1].copy()
        combined_vision = np.hstack((left_eye, right_eye))
        
        # --- 1. Dessin des contours ---
        for c in contours:
            try: cv2.drawContours(combined_vision, [c[0]], -1, (0, 0, 255), 2)
            except Exception: pass
        for c in dragonfly_contours:
            try: cv2.drawContours(combined_vision, [c], -1, (0, 255, 0), 2)
            except Exception: pass
        if best_contour is not None:
            try: cv2.drawContours(combined_vision, [best_contour[0]], -1, (255, 0, 0), 2)
            except Exception: pass

        # --- 2. Récupération des données ---
        debug_lines = [
            f"--- GENERAL ---",
            f"Step: {step}",
            f"Obstacles: {len(contours)} | Dragonflies: {len(dragonfly_contours)}",
            f"Best contour found: {'Yes' if best_contour is not None else 'No'}",
            ""
        ]

        # --- STATE (String) & DRIVES (Array) ---
        debug_lines.append("--- STATE & DRIVES ---")
        debug_lines.append(f"State: {state}")
        
        try:
            # Formatage du tableau drive (ex: [forward, turn])
            if isinstance(drive, (list, tuple, np.ndarray)) and len(drive) >= 2:
                debug_lines.append(f"Drive left : {drive[0]:.3f}")
                debug_lines.append(f"Drive right: {drive[1]:.3f}")
            else:
                debug_lines.append(f"Drives: {drive}")
        except Exception as e:
            debug_lines.append(f"Drive err: {type(e).__name__}")
        debug_lines.append("")
        
        try:
            
            debug_lines.append(f"Forces : {forces:.3f}")
            
        except Exception as e:
            debug_lines.append(f"Forces err: {type(e).__name__}")
        debug_lines.append("")

        # --- POSITION ---
        debug_lines.append("--- POSITION ---")
        try:
            pos = None
            if hasattr(sim, 'physics') and hasattr(sim.physics, 'data'):
                pos = sim.physics.data.qpos[:3]
            elif hasattr(sim, 'data') and hasattr(sim.data, 'qpos'):
                pos = sim.data.qpos[:3]
            else:
                pos = sim.fly.physics.data.qpos[:3] 
            
            if pos is not None:
                debug_lines.append(f"X: {pos[0]:.3f} | Y: {pos[1]:.3f} | Z: {pos[2]:.3f}")
            else:
                debug_lines.append("Position unreadable")
        except Exception as e:
            debug_lines.append(f"Pos err: {type(e).__name__}")
        debug_lines.append("")

        # --- OLFACTION ---
        debug_lines.append("--- OLFACTION ---")
        try:
            olf = sim.get_olfaction(sim.fly.name)
            olf_arr = np.array(olf)
            if olf_arr.ndim == 1 and len(olf_arr) >= 2:
                debug_lines.append(f"Left  : {olf_arr[0]:.7f}")
                debug_lines.append(f"Right : {olf_arr[1]:.7f}")
            else:
                val_str = np.array2string(olf_arr, precision=4, separator=', ', suppress_small=True)
                if len(val_str) > 38: val_str = val_str[:35] + "..."
                debug_lines.append(f"Raw: {val_str}")
        except Exception as e:
            debug_lines.append(f"Olf err: {type(e).__name__}")
        debug_lines.append("")

        # --- ANTENNES ---
        debug_lines.append("--- ANTENNA / WIND ---")
        try:
            data = sim.get_antenna_data(sim.fly.name)
            qpos_l = data['l']['qpos']
            theta_l = 2 * np.acos(qpos_l[0])
            axis_l = qpos_l[1:] / (np.sin(theta_l / 2) + 1e-6) 
            
            qpos_r = data['r']['qpos']
            theta_r = 2 * np.acos(qpos_r[0])
            axis_r = qpos_r[1:] / (np.sin(theta_r / 2) + 1e-6)
            
            debug_lines.extend([
                f"L-Angle: {theta_l:.2f} | Ax: [{axis_l[0]:.1f}, {axis_l[1]:.1f}, {axis_l[2]:.1f}]",
                f"R-Angle: {theta_r:.2f} | Ax: [{axis_r[0]:.1f}, {axis_r[1]:.1f}, {axis_r[2]:.1f}]"
            ])
        except Exception as e:
            debug_lines.append(f"Ant err: {type(e).__name__}")

        # --- 3. Adaptation de la taille de l'image ---
        required_height = max(combined_vision.shape[0], len(debug_lines) * LINE_SPACING + 30)
        
        if required_height > combined_vision.shape[0]:
            padding = np.zeros((required_height - combined_vision.shape[0], combined_vision.shape[1], 3), dtype=np.uint8)
            combined_vision_padded = np.vstack((combined_vision, padding))
        else:
            combined_vision_padded = combined_vision

        side_panel = np.zeros((required_height, PANEL_WIDTH, 3), dtype=np.uint8)

        y_offset = 25
        for line in debug_lines:
            cv2.putText(side_panel, line, (10, y_offset), FONT, FONT_SCALE, FONT_COLOR, 1, cv2.LINE_AA)
            y_offset += LINE_SPACING

        final_frame = np.hstack((combined_vision_padded, side_panel))
        writer.append_data(final_frame)

writer.close()
sim.renderer.show_in_notebook()
display(Video(video_filename, embed=True, width=1000))

 21%|██        | 10450/50000 [01:14<03:55, 168.26it/s]

Attractive intensities left:  1.089662943735396e-32
Attractive intensities right:  1.0788943872870727e-32


100%|██████████| 50000/50000 [22:13<00:00, 37.50it/s] 


nmf/backcam This browser does not support the video tag.


In [ ]:

# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# def detect_grass_obstacles(rgb_vision_array):
#     """
#     Detects green grass obstacles in an RGB image array and draws bounding boxes.
#     """
#     # 1. Convert the image from RGB to HSV color space
#     hsv_image = cv2.cvtColor(rgb_vision_array, cv2.COLOR_RGB2HSV)

#     plt.imshow(hsv_image)
#     plt.title("HSV Image")
#     plt.axis('off')
#     plt.show()

#     #do a canny edge detection to see the edges of the grass
#     edges = cv2.Canny(hsv_image, 100, 200)
#     plt.imshow(edges, cmap='gray')
#     plt.title("Canny Edges")
#     plt.axis('off')
#     plt.show()
#     # 2. Define the color range for the green grass in HSV
#     # You might need to tweak these numbers slightly, but this is a standard green range
#     lower_green = np.array([35, 50, 50])
#     upper_green = np.array([85, 255, 255])
    
#     # 3. Create a binary mask (White = Grass, Black = Everything else)
#     mask = cv2.inRange(hsv_image, lower_green, upper_green)
    
#     # 4. Find the contours (outlines) of the white patches in the mask
#     contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     # Make a copy of the original image so we can draw on it
#     annotated_image = rgb_vision_array.copy()
#     obstacle_positions = []
    
#     # 5. Loop through the detected grass objects
#     for contour in contours:
#         # Optional: Ignore very small specs of green (noise)
#         if cv2.contourArea(contour) > 50: 
            
#             # Get the bounding box coordinates for the grass
#             x, y, w, h = cv2.boundingRect(contour)
            
#             # Draw a bright red rectangle around the detected obstacle
#             cv2.rectangle(annotated_image, (x, y), (x + w, y + h), (255, 0, 0), 2)
            
#             # Save the center position of the obstacle (useful for your controller!)
#             center_x = x + (w // 2)
#             obstacle_positions.append(center_x)
            
#     return annotated_image, obstacle_positions

# # --- Test it on your stitched frame ---
# # Assuming 'stitched_frame' is the numpy array from your previous video code
# annotated_frame, positions = detect_grass_obstacles(raw_vision[0])  # Use the left eye's vision for detection

# plt.imshow(annotated_frame)
# plt.title("Grass Detection")
# plt.show()


# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# import cv2
# import numpy as np

# def detect_triangles_canny(rgb_vision_array):
#     """
#     Detects triangular shapes using Canny edge detection.
#     """
#     #use only the green channel for edge detection since the grass is green and we want to find the edges of the grass
#     green_channel = rgb_vision_array[:, :, 1]  # Extract the green channel
#     augment_contrast = cv2.equalizeHist(green_channel)  # Enhance contrast for better edge detection
#     edges = cv2.Canny(augment_contrast, 50, 150)
#     #crop to the center right portion of the image where the triangles are more likely to be
#     height, width = edges.shape
#     #keep only one third of the vertical height and the right half of the image

#     cropped_edges = edges[height//3:2*height//3, width//2:]  # Keep only the right half of the image
    
#     plt.imshow(cropped_edges, cmap='gray')
#     plt.title("Canny Edges (Cropped)")
#     plt.axis('off')
#     plt.show()
    
#     # 4. Find contours directly from the Canny edges
#     contours, _ = cv2.findContours(cropped_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     annotated_image = rgb_vision_array.copy()
#     triangle_centers = []
    
#     for contour in contours:
#         # Ignore small artifacts and noise
#         if cv2.contourArea(contour) > 10: 
            
#             # Calculate perimeter and approximate the shape
#             perimeter = cv2.arcLength(contour, True)
#             approx_polygon = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
            
#             # If the shape has exactly 3 corners, it's a triangle
#             if len(approx_polygon) == 3:
                
#                 # Draw the triangle outline in bright red
#                 cv2.drawContours(annotated_image, [approx_polygon], 0, (255, 0, 0), 2)
                
#                 # Calculate the center point
#                 M = cv2.moments(contour)
#                 if M['m00'] != 0:
#                     center_x = int(M['m10'] / M['m00'])
#                     triangle_centers.append(center_x)
                    
#     return annotated_image, triangle_centers


# import cv2
# import numpy as np
# import cv2
# import numpy as np
# import random
# import matplotlib.pyplot as plt

# def color_delimited_zones(canny_edges):
#     """
#     Prend une image issue d'un filtre Canny (noir et blanc) et 
#     remplit les zones fermées avec des couleurs aléatoires.
#     """
#     # 1. Dilatation légère (Optionnel mais recommandé)
#     # Cela permet de s'assurer que les lignes de contour sont bien fermées
#     kernel = np.ones((3, 3), np.uint8)
#     closed_edges = cv2.dilate(canny_edges, kernel, iterations=1)
    
#     # 2. Trouver les contours des objets
#     # RETR_EXTERNAL permet de ne prendre que l'enveloppe extérieure des objets
#     contours, _ = cv2.findContours(closed_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     # 3. Créer une toile de fond noire en couleur (3 canaux RGB)
#     height, width = canny_edges.shape
#     colored_image = np.zeros((height, width, 3), dtype=np.uint8)
    
#     # 4. Parcourir chaque contour et le colorier
#     for contour in contours:
        
#         # Ignorer les tout petits contours (le bruit)
#         if cv2.contourArea(contour) > 200: 
            
#             # Générer une couleur RGB aléatoire (entre 50 et 255 pour qu'elle soit visible)
#             color = (
#                 random.randint(50, 255), 
#                 random.randint(50, 255), 
#                 random.randint(50, 255)
#             )
            
#             # L'astuce est ici : thickness=cv2.FILLED (ou -1) indique à OpenCV 
#             # de colorier l'INTÉRIEUR de la zone, pas juste de tracer la ligne.
#             cv2.drawContours(colored_image, [contour], 0, color, thickness=cv2.FILLED)
            
#     return colored_image


# import cv2
# import numpy as np
# import random
# import matplotlib.pyplot as plt

# def fill_canny_edges(canny_image):
#     """
#     Takes a Canny edge map (black and white), closes gaps, 
#     and fills the enclosed shapes with random colors.
#     """
#     # 1. Close any microscopic gaps in the white edges
#     # This prevents the "paint" from leaking outside the shapes
#     kernel = np.ones((5, 5), np.uint8)
#     closed_edges = cv2.morphologyEx(canny_image, cv2.MORPH_CLOSE, kernel)
    
#     # 2. Find the connected contours
#     contours, _ = cv2.findContours(closed_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     # 3. Create a blank color canvas with the same dimensions as the input
#     height, width = canny_image.shape[:2]
#     filled_image = np.zeros((height, width, 3), dtype=np.uint8)
    
#     # Optional: If you want to keep the original white outlines visible 
#     # underneath the colors, uncomment the line below:
#     # filled_image[canny_image > 0] = [255, 255, 255]
    
#     # 4. Fill the insides!
#     for contour in contours:
        
#         # Filter out tiny specks of noise
#         if cv2.contourArea(contour) > 200: 
            
#             # Generate a random bright color (Red, Green, Blue)
#             color = (
#                 random.randint(100, 255), 
#                 random.randint(100, 255), 
#                 random.randint(100, 255)
#             )
            
#             # Draw and fill the inside of the contour
#             cv2.drawContours(filled_image, [contour], 0, color, thickness=cv2.FILLED)
            
#     return filled_image

# # --- How to use it ---
# # Assuming 'edges' is the Canny image you uploaded earlier:
# # colored_blocks = fill_canny_edges(edges)
# # 
# # plt.figure(figsize=(10, 5))
# # plt.imshow(colored_blocks)
# # plt.title("Filled Shapes from Canny Image")
# # plt.axis('off')
# # plt.show()

# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# def color_all_closed_areas(canny_image):
#     """
#     Takes a Canny edge map and fills every individual enclosed pocket 
#     of space with a unique random color.
#     """
#     # 1. Close microscopic gaps in the edges to ensure areas are truly "sealed"
#     kernel = np.ones((3, 3), np.uint8)
#     closed_edges = cv2.morphologyEx(canny_image, cv2.MORPH_CLOSE, kernel)
    
#     # 2. Invert the image! 
#     # Edges become black (0) and the empty space becomes white (255)
#     inverted_space = cv2.bitwise_not(closed_edges)
    
#     # 3. Find Connected Components
#     # This assigns a unique integer to every isolated patch of white pixels
#     num_labels, labels = cv2.connectedComponents(inverted_space)
    
#     # 4. Generate random RGB colors for every single label
#     # np.random is much faster than looping through every shape
#     colors = np.random.randint(50, 255, size=(num_labels, 3), dtype=np.uint8)
    
#     # Label 0 represents the black pixels (which are our actual edges)
#     # Let's make the edges themselves white so they stand out between the colors
#     colors[0] = [255, 255, 255] 
    
#     # 5. Identify the "Outside Background" and make it black
#     # Assuming the very top-left pixel (0,0) is empty space outside the grass
#     background_label = labels[0, 0]
#     colors[background_label] = [0, 0, 0]
    
#     # 6. Apply the colors to the image map (Numpy vectorization magic)
#     colored_image = colors[labels]
    
#     return colored_image

# # --- How to use it ---
# # Assuming 'edges' is your Canny image array:
# # colored_cells = color_all_closed_areas(edges)
# # 
# # plt.figure(figsize=(10, 5))
# # plt.imshow(colored_cells)
# # plt.title("Individually Colored Closed Areas")
# # plt.axis('off')
# # plt.show()

# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# def color_closed_areas_no_edges(canny_image):
#     """
#     Takes a Canny edge map, fills enclosed spaces with colors, 
#     and hides the original edges so they blend into the background.
#     """
#     # 1. Close gaps
#     kernel = np.ones((3, 3), np.uint8)
#     closed_edges = cv2.morphologyEx(canny_image, cv2.MORPH_CLOSE, kernel)
    
#     # 2. Invert space
#     inverted_space = cv2.bitwise_not(closed_edges)
    
#     # 3. Find Connected Components
#     num_labels, labels = cv2.connectedComponents(inverted_space)
    
#     # 4. Generate random RGB colors
#     colors = np.random.randint(50, 255, size=(num_labels, 3), dtype=np.uint8)
    
#     # THE FIX: Make the edges black so they disappear into the background
#     colors[0] = [0, 0, 0] 
    
#     # 5. Make the outside background black as well
#     background_label = labels[0, 0]
#     colors[background_label] = [0, 0, 0]
    
#     # 6. Apply colors
#     colored_image = colors[labels]
#     kernel = np.ones((5, 5), np.uint8)
#     colored_image = cv2.dilate(colored_image, kernel, iterations=1)
#     return colored_image

# # --- How to use it ---
# # colored_cells_clean = color_closed_areas_no_edges(edges)
# # 
# # plt.figure(figsize=(10, 5))
# # plt.imshow(colored_cells_clean)
# # plt.title("Colored Areas (Edges Hidden)")
# # plt.axis('off')
# # plt.show()

# def detect_obstacles_from_edges(rgb_vision_array):
#     """
#     Detects obstacles by finding large contours in a Canny edge map.
#     """
#     #crop to the center right portion of the image where the triangles are more likely to be
#     height, width, _ = rgb_vision_array.shape
#     rgb_vision_array = rgb_vision_array[height//3:2*height//3,  width//2:]
#     # 1. Standard Canny Pipeline
    
#     #use only the green channel for edge detection since the grass is green and we want to find the edges of the grass
#     green_channel = rgb_vision_array[:, :, 1]  # Extract the green channel
#     augment_contrast = cv2.equalizeHist(green_channel)  # Enhance contrast for better edge detection
#     edges = cv2.Canny(augment_contrast, 50, 150)
#     #crop to the center right portion of the image where the triangles are more likely to be
#     height, width = edges.shape
#     #keep only one third of the vertical height and the right half of the image

#      # Keep only the right half of the image
    
    
#     # 2. DILATION (The Secret Sauce)
#     # This thickens the white lines in your edge map to close any small gaps.
#     # It turns broken outlines into solid, enclosed shapes.
#     kernel = np.ones((3, 3), np.uint8)
#     solid_edges = cv2.dilate(edges, kernel, iterations=1)
#     plt.imshow(color_closed_areas_no_edges(solid_edges))
#     plt.title("Dilated Edges")
#     plt.axis('off')
#     plt.show()
#     # 3. Find the contours on the solid edges
#     contours, _ = cv2.findContours(solid_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     annotated_image = rgb_vision_array.copy()
#     obstacle_centers = []
    
#     # 4. Filter and Draw
#     for contour in contours:
        
#         # Get the straight bounding rectangle for the contour
#         x, y, w, h = cv2.boundingRect(contour)
        
#         # Filter out tiny specs of noise by checking the area of the bounding box
#         # You can adjust this '500' value. If it's picking up dust, make it higher.
#         if (w * h) > 500: 
            
#             # Draw a bright red rectangle around the obstacle
#             cv2.rectangle(annotated_image, (x, y), (x + w, y + h), (255, 0, 0), 2)
            
#             # Calculate and store the center X coordinate
#             center_x = x + (w // 2)
#             obstacle_centers.append(center_x)
            
#     return annotated_image, obstacle_centers


# # --- How to test it ---
# annotated_frame, positions = detect_obstacles_from_edges(raw_vision[0])  # Use the left eye's vision for detection
# plt.imshow(annotated_frame)
# plt.title("Obstacle Detection")
# plt.show()



#show raw vision output in notebook

